# Flood Risk Prediction — Kerala Aug 2018
**Demo Notebook** — Thin driver that imports modules and walks through the pipeline step-by-step.

No duplicated logic lives here. All computation is in `src/` modules.

In [ ]:
import sys
sys.path.insert(0, '..')

from config.aoi_config import (
    AOI_REGIONS, ACTIVE_REGION, VALIDATION_DATES, DATA_WINDOW,
    THRESHOLDS, WEIGHTS, get_region_name, get_aoi_bbox,
)

print(f"Region: {get_region_name()}")
print(f"AOI: {get_aoi_bbox()}")
print(f"Data window: {DATA_WINDOW['start']} to {DATA_WINDOW['end']}")
print(f"Validation dates: {list(VALIDATION_DATES.values())}")
print(f"Thresholds: {THRESHOLDS}")
print(f"Weights: {WEIGHTS}")

## Step 1: Data Ingestion
Fetch Sentinel-1 SAR, Sentinel-2 optical, SRTM DEM, and CHIRPS rainfall from Google Earth Engine.

In [ ]:
from src.ingestion.sentinel1 import init_gee, fetch_sar
from src.ingestion.sentinel2 import fetch_optical
from src.ingestion.dem import fetch_dem
from src.ingestion.rainfall import fetch_rainfall

init_gee()

sar_files = fetch_sar()
optical_files = fetch_optical()
dem_file = fetch_dem()
rainfall_files = fetch_rainfall()

print(f"\nIngested: {len(sar_files)} S1 scenes, {len(optical_files)} S2 scenes, "
      f"1 DEM, {len(rainfall_files)} rainfall days")

## Step 2: Preprocessing
Convert SAR to dB, compute NDWI, derive slope from DEM, align all to common grid.

In [ ]:
import numpy as np
import rasterio
from pathlib import Path
from src.preprocessing.sar import preprocess_sar
from src.preprocessing.optical import preprocess_optical
from src.preprocessing.terrain import preprocess_dem
from src.preprocessing.alignment import align_layers
from config.aoi_config import DATA_DIR

# Precompute static DEM-derived slope
slope_risk, elevation, slope_profile = preprocess_dem(dem_file)
print(f"DEM slope: {slope_risk.shape}, range: {slope_risk.min():.2f} - {slope_risk.max():.2f}")

# Process one example date
example_date = sorted(sar_files.keys())[len(sar_files)//2]  # middle date
print(f"\nPreprocessing example: {example_date}")

sar_db, sar_profile = preprocess_sar(sar_files[example_date])
print(f"  SAR dB: {sar_db.shape}, range: {sar_db.min():.1f} to {sar_db.max():.1f}")

ndwi, ndwi_profile = preprocess_optical(optical_files[example_date]['all_bands'])
print(f"  NDWI: {ndwi.shape}, range: {ndwi.min():.3f} to {ndwi.max():.3f}")

with rasterio.open(rainfall_files[example_date]) as src:
    rainfall = src.read(1).astype(np.float32)
    rain_profile = src.profile.copy()

aligned = align_layers(sar_db, ndwi, slope_risk, rainfall,
                       sar_profile, ndwi_profile, slope_profile, rain_profile)
print(f"  Aligned grid: {aligned['sar'].shape}")

## Step 3: Risk Scoring (MVP Threshold)
Weighted composite of water detection + terrain susceptibility + rainfall trigger.

In [ ]:
from src.scoring.threshold import compute_risk, water_detection, rainfall_trigger

risk = compute_risk(
    sar_db=aligned['sar'],
    ndwi=aligned['ndwi'],
    slope_risk=aligned['slope'],
    rainfall_3day=aligned['rainfall'],
    rainfall_7day=aligned['rainfall'],
)

water, conf = water_detection(aligned['sar'], aligned['ndwi'])

print(f"Risk map for {example_date}:")
print(f"  Shape: {risk.shape}")
print(f"  Mean: {risk.mean():.3f}")
print(f"  Max:  {risk.max():.3f}")
print(f"  Water pixels: {water.sum():.0f} / {water.size}")

# Visualize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sar_db, cmap='gray', vmin=-25, vmax=-5)
axes[0].set_title('SAR VH (dB)')
axes[1].imshow(ndwi, cmap='RdYlBu', vmin=-0.5, vmax=0.5)
axes[1].set_title('NDWI')
im = axes[2].imshow(risk, cmap='YlOrRd', vmin=0, vmax=1)
axes[2].set_title(f'Risk Score — {example_date}')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.savefig('../output/validation_charts/example_risk_map.png', dpi=150)
plt.show()
print("Saved: output/validation_charts/example_risk_map.png")

## Step 4: Full Time Series + Validation
Run scoring across all dates, plot risk over time, compare against flood dates.

In [ ]:
from src.scoring.threshold import compute_risk, save_risk_raster
from src.validation.validate import (
    plot_validation, compute_validation_stats, print_validation_report,
)
from config.aoi_config import OUTPUT_DIR

risk_timeseries = {}

for date_str in sorted(sar_files.keys()):
    # Preprocess
    sar_db, sar_p = preprocess_sar(sar_files[date_str])
    if date_str in optical_files:
        ndwi, ndwi_p = preprocess_optical(optical_files[date_str]['all_bands'])
    else:
        ndwi = np.zeros_like(sar_db)
        ndwi_p = sar_p
    
    # 3-day / 7-day rainfall accumulation
    from datetime import datetime, timedelta
    target_dt = datetime.strptime(date_str, '%Y-%m-%d')
    rain_3d = None
    rain_7d = None
    for delta in range(7):
        check = (target_dt - timedelta(days=delta)).strftime('%Y-%m-%d')
        if check in rainfall_files:
            with rasterio.open(rainfall_files[check]) as src:
                daily = src.read(1).astype(np.float32)
                if delta < 3:
                    rain_3d = daily if rain_3d is None else rain_3d + daily
                rain_7d = daily if rain_7d is None else rain_7d + daily
    rain_3d = rain_3d if rain_3d is not None else np.zeros_like(sar_db)
    rain_7d = rain_7d if rain_7d is not None else np.zeros_like(sar_db)
    
    aligned = align_layers(sar_db, ndwi, slope_risk, rain_3d,
                           sar_p, ndwi_p, slope_profile, rain_profile)
    
    risk = compute_risk(
        sar_db=aligned['sar'], ndwi=aligned['ndwi'],
        slope_risk=aligned['slope'],
        rainfall_3day=rain_3d, rainfall_7day=rain_7d,
    )
    
    risk_timeseries[date_str] = float(risk.mean())
    save_risk_raster(risk, aligned['profile'],
                     OUTPUT_DIR / 'risk_maps' / f'risk_{date_str}.tif',
                     satellite_date=date_str)
    print(f"  {date_str}: mean={risk.mean():.3f} max={risk.max():.3f}")

# Validation chart
chart_path = plot_validation(risk_timeseries)
stats = compute_validation_stats(risk_timeseries)
print_validation_report(stats)

## Step 5: Alerts
Flag regions exceeding the risk threshold.

In [ ]:
from src.alerts.alert import (
    flag_high_risk, save_alerts, compute_alert_summary, print_alert_report,
)

peak_date = stats.get('peak_date')
if peak_date:
    with rasterio.open(OUTPUT_DIR / 'risk_maps' / f'risk_{peak_date}.tif') as src:
        risk_peak = src.read(1)
        peak_profile = src.profile.copy()
    
    alerts = flag_high_risk(risk_peak, peak_profile)
    save_alerts(alerts, satellite_date=peak_date)
    summary = compute_alert_summary(alerts, risk_peak.size)
    print_alert_report(summary)
    
    if alerts:
        print(f"\nTop 5 highest-risk locations:")
        for a in alerts[:5]:
            print(f"  ({a['lat']:.4f}, {a['lon']:.4f}): risk={a['risk']:.3f}")
else:
    print("No peak date found.")

## Output Summary
All generated files and data disclosure.

In [ ]:
from pathlib import Path

output_dir = OUTPUT_DIR
print("Generated outputs:")
print(f"  Risk maps:     {output_dir / 'risk_maps'}")
for f in sorted((output_dir / 'risk_maps').glob('*.tif')):
    print(f"    {f.name}")

print(f"  Validation:    {output_dir / 'validation_charts'}")
for f in sorted((output_dir / 'validation_charts').glob('*.png')):
    print(f"    {f.name}")

print(f"  Alerts:        {output_dir / 'alerts'}")
for f in sorted((output_dir / 'alerts').glob('*.csv')):
    print(f"    {f.name}")

print(f"\nData as of satellite pass dates.")
print(f"Next pass expected per Sentinel-1 revisit schedule (~6 days).")